# 08 - Topic-Level Watermark Detection

Detector for the **first layer** (public, topic-based watermark) built in
`topic_wise_gen_pipeline.ipynb` / `src/watermark/first_layer.py`.

## The scheme being detected
1. `extract_topic(prompt)` — mean-pool the prompt's token embeddings in OPT's own
   embedding space, cosine-compare against one vector per predetermined topic, take argmax.
2. Boost that topic's **fixed greenlist** logits by `delta` at *every* generation step.

## Detection as a hypothesis test
Because the greenlist is fixed per topic (not context-dependent like KGW), every
scored token is an independent Bernoulli trial:

- **H0** (text is NOT watermarked): each token lands in topic `t`'s greenlist with probability  
  `gamma_t = |greenlist_t| / vocab_size`
- **H1** (watermarked with delta > 0): green tokens are over-represented

Given `s` green hits among `n` scored tokens:

```
z = (s/n - gamma) / sqrt(gamma*(1-gamma)/n)
p_value = P(Binom(n, gamma) >= s)   # survival function, upper tail
```

Verdict = watermarked iff `z >= Z_THRESHOLD` AND `p_value < P_VALUE_THRESHOLD`.

> Same statistical core as `src/detection/kgw_detection.py`, but the green set is
> looked up from a public CSV via the inferred topic instead of derived from a secret key.

## Setup — Colab VM only

Clone the repo, enter it, install runtime deps, log in to HF.
(Nothing here is meant to run on a local machine.)

In [1]:
!git clone https://github.com/pravaspaudel/Dual_watermarking_Scheme.git
%cd /content/Dual_watermarking_Scheme

Cloning into 'Dual_watermarking_Scheme'...
remote: Enumerating objects: 163, done.
remote: Counting objects: 100% (12/12), done.
remote: Compressing objects: 100% (12/12), done.
remote: Total 163 (delta 0), reused 5 (delta 0), pack-reused 151 (from 1)
Receiving objects: 100% (163/163), 42.77 MiB | 34.46 MiB/s, done.
Resolving deltas: 100% (59/59), done.
/content/Dual_watermarking_Scheme


In [2]:
!pip install transformers scipy pandas

In [3]:
import os
import torch
from huggingface_hub import login
login()

device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
  warnings.warn(f"\nError while fetching `HF_TOKEN` secret value from your vault: '{str(e)}'.")


device: cuda


In [6]:
# ------------------------------------------------------------------
# Locate topic_greenlists.csv (works from notebooks/ or repo root or Colab)
# ------------------------------------------------------------------
def find_file(name):
    
    p = os.path.join(name)
    if os.path.isfile(p):
        return p
    raise FileNotFoundError(f"{name} not found under")

GREENLIST_CSV = find_file("../topic_greenlists.csv")
print("greenlists:", GREENLIST_CSV)

greenlists: ../topic_greenlists.csv


## Step 1 - Load the public topic greenlists

Identical source of truth as the generation side: `topic_greenlists.csv`
(one row per greenlisted token: `topic, token_id, token, similarity`).

The greenlist is **public and fixed per topic**, so detection needs no secret key -
only the CSV, the vocab size, and the ability to re-infer the topic.

In [7]:
import pandas as pd

topic_greenlist = pd.read_csv(GREENLIST_CSV)

TOPICS = sorted(topic_greenlist["topic"].unique())
greenlists = {t: g["token_id"].astype(int).tolist()
              for t, g in topic_greenlist.groupby("topic")}

print("topics:", TOPICS)
print("greenlist sizes:", {t: len(ids) for t, ids in greenlists.items()})
topic_greenlist.head()

topics: ['entertainment', 'finance', 'history', 'medicine', 'politics', 'science', 'sports', 'technology']
greenlist sizes: {'entertainment': 110, 'finance': 102, 'history': 1670, 'medicine': 95, 'politics': 3566, 'science': 1958, 'sports': 2575, 'technology': 7105}


,topic,token_id,token,similarity
0,technology,806,Ġtechnology,0.6282
1,technology,2903,Ġtech,0.4063
2,technology,3165,Ġtechnical,0.1689
3,technology,3777,ĠTechnology,0.6306
4,technology,3815,ĠNIGHT,0.1886


## Step 2 - Model, embeddings, topic matrix

Same geometry that built the greenlists (`04_cosine_similarity.ipynb`) and drove
generation (`topic_wise_gen_pipeline.ipynb`):

- unit-normalize OPT's input embedding table -> a dot product between rows **is** a cosine
- topic vector = embedding of the first subtoken of `" " + topic` (e.g. `Ġtechnology`)

Detection must use the **same** embedding space as generation, otherwise the inferred
topic can drift away from the one used to pick the greenlist.

In [8]:
from transformers import AutoTokenizer, AutoModelForCausalLM

MODEL_NAME = "facebook/opt-2.7b"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(MODEL_NAME).to(device)
model.eval()

VOCAB_SIZE = len(tokenizer)  
print("vocab_size:", VOCAB_SIZE)

config.json:   0%|          | 0.00/691 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/685 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/441 [00:00<?, ?B/s]

...

Loading weights:   0%|          | 0/517 [00:00<?, ?it/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 5.30GB            

model.safetensors: downloading bytes:           |  0.00B            

generation_config.json:   0%|          | 0.00/137 [00:00<?, ?B/s]

vocab_size: 50265


In [9]:
# unit-normalized input embeddings: rows live on the unit sphere
embedding_matrix = model.get_input_embeddings().weight.detach().to(device)
normed_embeddings = embedding_matrix / embedding_matrix.norm(dim=1, keepdim=True)

topic_token_ids = [tokenizer.encode(" " + t, add_special_tokens=False)[0] for t in TOPICS]
for t, tid in zip(TOPICS, topic_token_ids):
    print(f"{t:14} -> {tokenizer.decode([tid])!r} (id {tid})")

topic_matrix = normed_embeddings[topic_token_ids]        # (K, d), unit rows
print("topic_matrix:", tuple(topic_matrix.shape))

entertainment  -> ' entertainment' (id 4000)
finance        -> ' finance' (id 2879)
history        -> ' history' (id 750)
medicine       -> ' medicine' (id 6150)
politics       -> ' politics' (id 2302)
science        -> ' science' (id 2866)
sports         -> ' sports' (id 1612)
technology     -> ' technology' (id 806)
topic_matrix: (8, 2560)


## Step 3 - Re-infer the topic from the text under test

At detection time we only have the text (the prompt may be unknown), so we run the
**same** `extract_topic` logic on the generated text itself. The boosted tokens pull
their own topic to the top, which is exactly what routes us to the correct greenlist.

In [10]:
@torch.no_grad()
def extract_topic(text: str, verbose: bool = False):
    """Mean-pool token embeddings of `text`, cosine vs each topic vector.

    Same logic as first_layer.TopicWiseWatermarking.extract_topic.
    Returns (best_topic, ranked list of (topic, cosine)).
    """
    ids = tokenizer.encode(text, add_special_tokens=False)
    if len(ids) == 0:
        return None, []

    vec = normed_embeddings[ids].mean(dim=0)
    vec = vec / vec.norm()

    scores = (topic_matrix @ vec).tolist()          # (K,) cosine per topic
    ranked = sorted(zip(TOPICS, scores), key=lambda x: -x[1])

    if verbose:
        for name, s in ranked:
            bar = "#" * max(0, int((s + 1) * 20))   # cosine in [-1, 1] -> bar
            print(f"  {name:14} {s:+.4f} {bar}")
    return ranked[0][0], ranked

In [11]:
# quick feel for the geometry on a few off-the-shelf texts
for p in [
    "The government passed a new law before the election",
    "The cricket team won the final match",
]:
    print(f"\n{p!r}")
    best, _ = extract_topic(p, verbose=True)
    print("->", best)


'The government passed a new law before the election'
  history        +0.2499 ########################
  technology     +0.2312 ########################
  sports         +0.1985 #######################
  science        +0.1877 #######################
  politics       +0.1722 #######################
  entertainment  +0.1165 ######################
  medicine       +0.1006 ######################
  finance        +0.0933 #####################
-> history

'The cricket team won the final match'
  sports         +0.2668 #########################
  history        +0.2147 ########################
  science        +0.1877 #######################
  technology     +0.1853 #######################
  politics       +0.1151 ######################
  entertainment  +0.1041 ######################
  finance        +0.0982 #####################
  medicine       +0.0955 #####################
-> sports


## Step 4 - The detection statistic

For topic `t` with greenlist `G_t` (`gamma_t = |G_t| / vocab_size`) and a text of `n`
scored tokens containing `s` green hits:

```
green_rate   = s / n
expected     = n * gamma_t
std          = sqrt(n * gamma_t * (1 - gamma_t))
z            = (s - expected) / std          ~ N(0, 1) under H0
p_value      = binomtest(s, n, gamma_t, alternative='greater').pvalue
```

Notes
- Unlike KGW there is **no context window / prev-token hash**: the greenlist is fixed,
  so *every* token position is scoreable (n is larger -> more power).
- `p_value` is exact binomial; z is its normal approximation. We report both because
  z is interpretable as "how many standard deviations above chance", while p is what
  the decision rule thresholds.
- Per-topic `gamma` varies wildly here (technology ~0.14 vs medicine ~0.002), so using
  one global green fraction would badly miscalibrate the test. Always use the topic's own gamma.

In [12]:
import math
from scipy.stats import binomtest, norm

Z_THRESHOLD = 4.0        # z must exceed this (KGW-style tau)
P_VALUE_THRESHOLD = 0.05 # ... and the upper-tail binomial p-value must be below this

# precompute per-topic green sets + fractions once
green_sets = {t: frozenset(ids) for t, ids in greenlists.items()}
green_fractions = {t: len(ids) / VOCAB_SIZE for t, ids in greenlists.items()}
for t in TOPICS:
    print(f"{t:14} |G_t|={len(greenlists[t]):>6}  gamma_t={green_fractions[t]:.4f}")

entertainment  |G_t|=   110  gamma_t=0.0022
finance        |G_t|=   102  gamma_t=0.0020
history        |G_t|=  1670  gamma_t=0.0332
medicine       |G_t|=    95  gamma_t=0.0019
politics       |G_t|=  3566  gamma_t=0.0709
science        |G_t|=  1958  gamma_t=0.0390
sports         |G_t|=  2575  gamma_t=0.0512
technology     |G_t|=  7105  gamma_t=0.1414


In [13]:
def detection_stats(token_ids, topic):
    """Count green hits and compute z-score + exact binomial p-value."""
    gset = green_sets[topic]
    gamma = green_fractions[topic]

    hits = sum(1 for tid in token_ids if tid in gset)
    n = len(token_ids)

    if n == 0 or gamma == 0:
        return {"hits": hits, "num_positions": n, "gamma": gamma,
                "green_rate": 0.0, "z_score": 0.0, "p_value": 1.0}

    expected = n * gamma
    std = math.sqrt(n * gamma * (1 - gamma))
    z = (hits - expected) / std
    p_value = binomtest(k=hits, n=n, p=gamma, alternative="greater").pvalue

    return {"hits": hits, "num_positions": n, "gamma": round(gamma, 5),
            "green_rate": round(hits / n, 4),
            "z_score": round(z, 4), "p_value": p_value}

In [15]:
def detect_topic_watermark(text, z_threshold=Z_THRESHOLD,
                           p_value_threshold=P_VALUE_THRESHOLD, verbose=False):
    """Full first-layer detection: infer topic -> green hits -> z vs p decision.

    Returns dict (same spirit as detect_private_watermark in kgw_detection.py):
      topic, topic_score, ownership_score (= s/n), match_count, num_positions,
      gamma, z_score, p_value, confirmed
    """
    best_topic, ranked = extract_topic(text)
    if best_topic is None:
        return {"topic": None, "topic_score": None, "ownership_score": 0.0,
                "match_count": 0, "num_positions": 0, "gamma": 0.0,
                "z_score": 0.0, "p_value": 1.0, "confirmed": False}

    token_ids = tokenizer.encode(text, add_special_tokens=False)
    stats = detection_stats(token_ids, best_topic)

    confirmed = bool(stats["z_score"] >= z_threshold
                     and stats["p_value"] < p_value_threshold)

    result = {
        "topic": best_topic,
        "topic_score": round(ranked[0][1], 4),
        "ownership_score": stats["green_rate"],
        "match_count": stats["hits"],
        "num_positions": stats["num_positions"],
        "gamma": stats["gamma"],
        "z_score": stats["z_score"],
        "p_value": stats["p_value"],
        "confirmed": confirmed,
    }

    if verbose:
        print(f"inferred topic : {best_topic} (cosine {result['topic_score']:+.4f})")
        print(f"green hits     : {stats['hits']}/{stats['num_positions']}"
              f"  (expected {stats['gamma'] * stats['num_positions']:.1f}, gamma={stats['gamma']})")
        print(f"z-score        : {stats['z_score']:+.2f}  (threshold {z_threshold})")
        print(f"p-value        : {stats['p_value']:.2e}  (threshold {p_value_threshold})")
        print(f"watermarked?   : {'YES' if confirmed else 'no'}")

    return result

### Smoke test on hand-written text

Human/other-LLM text should score near `z ~ 0`, `p ~ 0.5` and NOT be flagged.
The two texts below also probe that topic routing lands on a *sensible* greenlist.

In [16]:
# unwatermarked human-ish text -> expect z near 0, p large, confirmed=False
for t in [
    "The quick brown fox jumps over the lazy dog near the river bank.",
    "She packed her bags, booked a flight and left without saying goodbye.",
]:
    print(f"\n{t!r}")
    _ = detect_topic_watermark(t, verbose=True)


'The quick brown fox jumps over the lazy dog near the river bank.'
inferred topic : history (cosine +0.2234)
green hits     : 0/14  (expected 0.5, gamma=0.03322)
z-score        : -0.69  (threshold 4.0)
p-value        : 1.00e+00  (threshold 0.05)
watermarked?   : no

'She packed her bags, booked a flight and left without saying goodbye.'
inferred topic : history (cosine +0.2336)
green hits     : 0/14  (expected 0.5, gamma=0.03322)
z-score        : -0.69  (threshold 4.0)
p-value        : 1.00e+00  (threshold 0.05)
watermarked?   : no


## Step 5 - Sanity check on saved generation outputs

`data/example/topic_wise_watermarked_example.csv` holds `plain_output` vs
`watermarked_output` pairs produced by the generation notebook. The detector should
separate them: high z / tiny p for watermarked, near-null stats for plain.

In [17]:

examples = pd.read_csv("/content/Dual_watermarking_Scheme/data/example/topic_wise_watermarked_example.csv")
print(examples[["prompt", "topic"]])
examples.head()

                                              prompt       topic
0  The government passed a new law before the ele...     history
1               The cricket team won the final match      sports
2                 The new AI chip runs twice as fast  technology


,id,prompt,topic,topic_score,plain_output,watermarked_output
0,0,The government passed a new law before the ele...,history,0.2512,The government passed a new law before the ele...,The government passed a new law before the ele...
1,1,The cricket team won the final match,sports,0.2634,The cricket team won the final match of the se...,The cricket team won the final match of the To...
2,2,The new AI chip runs twice as fast,technology,0.2389,The new AI chip runs twice as fast as a standa...,The new AI chip runs twice as fast as today's ...


In [18]:
rows = []
for _, r in examples.iterrows():
    for col, label in (("plain_output", "plain"), ("watermarked_output", "watermarked")):
        res = detect_topic_watermark(r[col])
        rows.append({
            "prompt": r["prompt"],
            "kind": label,
            "gen_topic": r["topic"],          # topic assigned at generation time
            "detected_topic": res["topic"],   # topic re-inferred by detector
            "hits": res["match_count"],
            "n": res["num_positions"],
            "gamma": res["gamma"],
            "green_rate": res["ownership_score"],
            "z_score": res["z_score"],
            "p_value": res["p_value"],
            "confirmed": res["confirmed"],
        })
sanity_df = pd.DataFrame(rows)
pd.set_option("display.width", 160)
sanity_df

,prompt,kind,gen_topic,detected_topic,hits,n,gamma,green_rate,z_score,p_value,confirmed
0,The government passed a new law before the ele...,plain,history,history,0,59,0.03322,0.0000,-1.4239,1.000000,False
1,The government passed a new law before the ele...,watermarked,history,history,3,59,0.03322,0.0508,0.7553,0.312346,False
2,The cricket team won the final match,plain,sports,history,0,57,0.03322,0.0000,-1.3996,1.000000,False
3,The cricket team won the final match,watermarked,sports,sports,11,57,0.05123,0.1930,4.8544,0.000131,True
4,The new AI chip runs twice as fast,plain,technology,technology,0,58,0.14135,0.0000,-3.0900,1.000000,False
5,The new AI chip runs twice as fast,watermarked,technology,technology,20,58,0.14135,0.3448,4.4481,0.000081,True


In [19]:
# topic routing check: did the detector recover the generation-time topic?
routing_ok = (sanity_df[sanity_df["kind"] == "watermarked"]["gen_topic"]
              == sanity_df[sanity_df["kind"] == "watermarked"]["detected_topic"])
print("topic recovered on watermarked texts:", f"{routing_ok.mean():.0%}")
print("\nmean z: plain = %.2f | watermarked = %.2f" % (
    sanity_df[sanity_df.kind == "plain"].z_score.mean(),
    sanity_df[sanity_df.kind == "watermarked"].z_score.mean()))

topic recovered on watermarked texts: 100%

mean z: plain = -1.97 | watermarked = 3.35


## Step 6 - Live end-to-end: generate -> detect

Fresh generations (not the saved CSV) using `first_layer.TopicBoostProcessor`,
sweeping `delta`. Expected behaviour:

- `delta = 0` behaves like the plain model -> z near 0
- larger delta pushes more tokens into the greenlist -> z climbs
- detection flips to `confirmed` once z crosses the threshold and p collapses

In [21]:
import sys
sys.path.insert(0, "src")       # running from repo root after %cd
sys.path.insert(0, "..")        # running from notebooks/

from src.watermark.first_layer import TopicBoostProcessor

In [22]:
torch.manual_seed(42)

PROMPT = "The government passed a new law before the election"
topic, _ = extract_topic(PROMPT)
print("assigned topic:", topic)

DELTA_SWEEP = [0.0, 1.0, 2.0, 4.0, 6.0]
sweep_rows = []
for d in DELTA_SWEEP:
    inputs = tokenizer(PROMPT, return_tensors="pt").to(model.device)
    processors = [TopicBoostProcessor(greenlists[topic], delta=d)] if d > 0 else None
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=80, do_sample=True,
                             temperature=1.0, top_p=0.9,
                             logits_processor=processors)
    text = tokenizer.decode(out[0], skip_special_tokens=True)

    det = detect_topic_watermark(text)
    sweep_rows.append({"delta": d, "detected_topic": det["topic"],
                       "hits": det["match_count"], "n": det["num_positions"],
                       "green_rate": det["ownership_score"],
                       "gamma": det["gamma"], "z_score": det["z_score"],
                       "p_value": det["p_value"], "confirmed": det["confirmed"]})
    print(f"delta={d:>4}  z={det['z_score']:+7.2f}  p={det['p_value']:.3e}"
          f"  confirmed={det['confirmed']}\n  {text[:110]}...")

sweep_df = pd.DataFrame(sweep_rows)
sweep_df

assigned topic: history
delta= 0.0  z=  -1.16  p=9.506e-01  confirmed=False
  The government passed a new law before the election giving people an automatic right to abortion in the first ...
delta= 1.0  z=  -1.75  p=1.000e+00  confirmed=False
  The government passed a new law before the election, giving them more leeway to take away people's rights with...
delta= 2.0  z=  -1.75  p=1.000e+00  confirmed=False
  The government passed a new law before the election that makes it a felony to own an untraceable firearm. The ...
delta= 4.0  z=  -0.57  p=7.994e-01  confirmed=False
  The government passed a new law before the election in which they could remove you from the electoral roll if ...
delta= 6.0  z= +23.09  p=7.677e-38  confirmed=True
  The government passed a new law before the election, and Hollande hesitated to sign it so as to not appear to ...


,delta,detected_topic,hits,n,green_rate,gamma,z_score,p_value,confirmed
0,0.0,history,1,89,0.0112,0.03322,-1.1574,9.505684e-01,False
1,1.0,history,0,89,0.0000,0.03322,-1.7489,1.000000e+00,False
2,2.0,history,0,89,0.0000,0.03322,-1.7489,1.000000e+00,False
3,4.0,history,2,89,0.0225,0.03322,-0.5660,7.993794e-01,False
4,6.0,history,42,89,0.4719,0.03322,23.0920,7.677089e-38,True


## Step 7 - Batch detection over a DataFrame

`detect_dataframe` mirrors `kgw_detection.detect_dataframe`: original columns +
one column per detection statistic. Run it over both columns so every text gets
a verdict.

In [23]:
def detect_dataframe(df, text_column="text", z_threshold=Z_THRESHOLD,
                     p_value_threshold=P_VALUE_THRESHOLD):
    """Return df with detection stats appended for each row's `text_column`."""
    results = [detect_topic_watermark(t, z_threshold, p_value_threshold)
               for t in df[text_column]]
    det = pd.DataFrame(results).add_prefix("det_")
    return pd.concat([df.reset_index(drop=True), det], axis=1)

texts = examples.rename(columns={"watermarked_output": "text"})[["prompt", "topic", "text"]]
detect_dataframe(texts)

,prompt,topic,text,det_topic,det_topic_score,det_ownership_score,det_match_count,det_num_positions,det_gamma,det_z_score,det_p_value,det_confirmed
0,The government passed a new law before the ele...,history,The government passed a new law before the ele...,history,0.2720,0.0508,3,59,0.03322,0.7553,0.312346,False
1,The cricket team won the final match,sports,The cricket team won the final match of the To...,sports,0.3132,0.1930,11,57,0.05123,4.8544,0.000131,True
2,The new AI chip runs twice as fast,technology,The new AI chip runs twice as fast as today's ...,technology,0.3792,0.3448,20,58,0.14135,4.4481,0.000081,True


## Step 8 - Threshold sweep: z vs p trade-off

The decision rule has two knobs. Sweeping the z-threshold shows the classic
trade-off: stricter thresholds miss weak watermarks (false negatives), looser
ones flag clean text (false positives). Under H0, z ~ N(0,1), so a threshold of
~2 already means ~2.3% FPR; 4 is very conservative.

In [24]:
from scipy.stats import norm

print("z_threshold -> false-positive rate under H0 (p = P(Z >= z))")
for zt in [1.64, 1.96, 2.33, 3.0, 4.0]:
    print(f"  {zt:4.2f} -> {1 - norm.cdf(zt):.4f}")

# how many of our example texts flip at each threshold?
rows = []
plain = sanity_df[sanity_df.kind == "plain"]
wm    = sanity_df[sanity_df.kind == "watermarked"]
for zt in [1.64, 1.96, 2.33, 3.0, 4.0]:
    rows.append({
        "z_threshold": zt,
        "plain_flagged": int((plain.z_score >= zt).sum()),
        "wm_detected": int((wm.z_score >= zt).sum()),
        "wm_total": len(wm),
    })
pd.DataFrame(rows)

z_threshold -> false-positive rate under H0 (p = P(Z >= z))
  1.64 -> 0.0505
  1.96 -> 0.0250
  2.33 -> 0.0099
  3.00 -> 0.0013
  4.00 -> 0.0000


,z_threshold,plain_flagged,wm_detected,wm_total
0,1.64,0,2,3
1,1.96,0,2,3
2,2.33,0,2,3
3,3.00,0,2,3
4,4.00,0,2,3


## Step 9 - Persist results

In [25]:
os.makedirs("data/example", exist_ok=True)
sanity_df.to_csv("data/example/topic_detection_sanity.csv", index=False)
sweep_df.to_csv("data/example/topic_detection_delta_sweep.csv", index=False)
print("saved data/example/topic_detection_sanity.csv")
print("saved data/example/topic_detection_delta_sweep.csv")

saved data/example/topic_detection_sanity.csv
saved data/example/topic_detection_delta_sweep.csv


## Step 10 - Reusable detector module

Freeze the notebook logic into `src/detection/topic_detection.py`, the first-layer
counterpart of `src/detection/kgw_detection.py`. A small class caches the unit-norm
embedding table + topic matrix once instead of rebuilding them per call.

In [ ]:
%%writefile src/detection/topic_detection.py
'''First-layer (public, topic-based) watermark detection.

Counterpart of kgw_detection.py for the topic-wise scheme: re-infers the topic
from the text via cosine similarity in OPT's embedding space, looks up that
topic's fixed greenlist from topic_greenlists.csv, counts green-token hits and
tests them against H0 rate gamma = |G|/vocab_size using a z-score and an exact
binomial upper-tail p-value.
'''
import math

import torch
import pandas as pd
from scipy.stats import binomtest

Z_THRESHOLD = 4.0
P_VALUE_THRESHOLD = 0.05


def load_greenlists(csv_path):
    """CSV(topic, token_id, ...) -> {topic: frozenset(token_ids)}."""
    df = pd.read_csv(csv_path)
    return {t: frozenset(g["token_id"].astype(int))
            for t, g in df.groupby("topic")}


def build_topic_matrix(model, tokenizer, topics):
    """(K, d) unit-norm rows: OPT input embedding of each topic's first subtoken."""
    emb = model.get_input_embeddings().weight.detach()
    emb = emb / emb.norm(dim=1, keepdim=True)
    ids = [tokenizer.encode(" " + t, add_special_tokens=False)[0] for t in topics]
    return emb[ids]


class TopicWatermarkDetector:
    """Detects watermarks made by first_layer.TopicWiseWatermarking.

    Use the SAME model/tokenizer/CSV as generation so the topic geometry and
    gamma values match.
    """

    def __init__(self, model, tokenizer, greenlist_csv,
                 z_threshold=Z_THRESHOLD, p_value_threshold=P_VALUE_THRESHOLD):
        self.model = model.eval()
        self.tokenizer = tokenizer
        self.vocab_size = len(tokenizer)
        self.z_threshold = z_threshold
        self.p_value_threshold = p_value_threshold

        self.green_sets = load_greenlists(greenlist_csv)
        self.topics = sorted(self.green_sets)
        self.green_fractions = {
            t: len(s) / self.vocab_size for t, s in self.green_sets.items()
        }
        self.topic_matrix = build_topic_matrix(model, tokenizer, self.topics)

        emb = model.get_input_embeddings().weight.detach()
        self.normed_embeddings = emb / emb.norm(dim=1, keepdim=True)

    @torch.no_grad()
    def extract_topic(self, text):
        """Mean-pool token embeddings, cosine vs each topic vector."""
        ids = self.tokenizer.encode(text, add_special_tokens=False)
        if not ids:
            return None, []
        vec = self.normed_embeddings[ids].mean(dim=0)
        vec = vec / vec.norm()
        scores = (self.topic_matrix @ vec).tolist()
        ranked = sorted(zip(self.topics, scores), key=lambda x: -x[1])
        return ranked[0][0], ranked

    def detection_stats(self, token_ids, topic):
        gset = self.green_sets[topic]
        gamma = self.green_fractions[topic]

        hits = sum(1 for tid in token_ids if tid in gset)
        n = len(token_ids)
        if n == 0 or gamma == 0:
            return hits, n, gamma, 0.0, 1.0

        z = (hits - n * gamma) / math.sqrt(n * gamma * (1 - gamma))
        p_value = binomtest(k=hits, n=n, p=gamma, alternative="greater").pvalue
        return hits, n, gamma, z, p_value

    def detect(self, text, verbose=False):
        best_topic, ranked = self.extract_topic(text)
        if best_topic is None:
            return {"topic": None, "topic_score": None, "ownership_score": 0.0,
                    "match_count": 0, "num_positions": 0, "gamma": 0.0,
                    "z_score": 0.0, "p_value": 1.0, "confirmed": False}

        token_ids = self.tokenizer.encode(text, add_special_tokens=False)
        hits, n, gamma, z, p_value = self.detection_stats(token_ids, best_topic)

        confirmed = bool(z >= self.z_threshold and p_value < self.p_value_threshold)

        if verbose:
            print(f"inferred topic : {best_topic} (cosine {ranked[0][1]:+.4f})")
            print(f"green hits     : {hits}/{n}  (expected {gamma * n:.1f}, gamma={gamma:.5f})")
            print(f"z-score        : {z:+.2f}  (threshold {self.z_threshold})")
            print(f"p-value        : {p_value:.2e}  (threshold {self.p_value_threshold})")
            print(f"watermarked?   : {'YES' if confirmed else 'no'}")

        return {
            "topic": best_topic,
            "topic_score": round(ranked[0][1], 4),
            "ownership_score": round(hits / n, 4) if n else 0.0,
            "match_count": hits,
            "num_positions": n,
            "gamma": round(gamma, 5),
            "z_score": round(z, 4),
            "p_value": p_value,
            "confirmed": confirmed,
        }

    def detect_dataframe(self, df, text_column="text"):
        results = [self.detect(t) for t in df[text_column]]
        det = pd.DataFrame(results).add_prefix("det_")
        return pd.concat([df.reset_index(drop=True), det], axis=1)